# Práctica NoSQL: Python, Dask y MongoDB Atlas

**Equipo:** Alejandro Alvarenga, Eduardo Navarrete, Josué Melara · **Fecha:** 23-octubre-2026

## Requisitos de entorno
- **Python 3.10, 3.11 o 3.12** (recomendado 3.11 o 3.12). La celda de configuración falla con un mensaje claro si la versión es menor que 3.10.
- Un entorno virtual propio, por ejemplo `python -m venv .venv`, activado antes de abrir Jupyter.
- Cuenta de MongoDB Atlas (solo para la Parte 3). La URI no se escribe en este cuaderno: va en un archivo `.env` local que no se sube al repositorio.

## Reparto del trabajo
| Parte | Rol | Responsable | Entrega |
|---|---|---|---|
| 0 | Configuración común | Todos | Entorno funcionando |
| 1 | Ingeniero de Datos | César Navarrete | Lógica Dask, limpieza, generación de datos, validación de filas |
| 2 | Arquitecto Cloud | [completar nombre] | Benchmark Pandas vs. Dask, entorno de ejecución, conectividad AWS/GCP |
| 3 | DBA NoSQL | Alejandro Alvarenga | Conexión Atlas, índice, upsert, 3 consultas |
| 4 | Registro y entrega | Todos | Aportes, commits, revisión cruzada, evidencia cloud |

## Revisión cruzada
| Parte revisada | Revisa |
|---|---|
| Parte 1 (Ingeniero de Datos) | DBA NoSQL |
| Parte 2 (Arquitecto Cloud) | Ingeniero de Datos |
| Parte 3 (DBA NoSQL) | Arquitecto Cloud |

Además, los tres ejecutan el cuaderno completo en su máquina.

**Orden de ejecución:** de arriba abajo. La Parte 3 necesita los archivos de `results/` que generan las Partes 1 y 2.

## Parte 0 · Preparación (todos)

### 0.1 Dependencias
Ejecuta esta celda **una vez** por entorno y luego **reinicia el kernel** (Kernel → Restart). Las versiones mínimas evitan incompatibilidades conocidas (cadenas de texto en Dask/Pandas, conexión `mongodb+srv://`, certificados TLS).

In [1]:
%pip install -q "dask[dataframe]>=2024.8" "pandas>=2.1" "numpy>=1.26" "pyarrow>=14" "pymongo>=4.6" "python-dotenv>=1.0" "certifi" "psutil"

Note: you may need to restart the kernel to use updated packages.


### 0.2 Configuración y funciones de apoyo
Aquí están todos los parámetros. Para una prueba rápida se puede bajar `RECORDS` (por ejemplo a `200_000`) y para un volumen mayor, subirlo. Si el volumen supera o no la RAM se determina con la medición de la Parte 2.

In [2]:
import sys, os, json, time, glob, platform, statistics
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import dask
import dask.dataframe as dd
import psutil

assert sys.version_info >= (3, 10), f"Se requiere Python 3.10 o superior; tienes {sys.version.split()[0]}"

# --- Parámetros (editables) ---
DATA_DIR    = Path('data')
RESULTS_DIR = Path('results')
RECORDS     = 2_000_000     # filas totales a generar
FILES       = 16            # número de CSV
GEN_CHUNK   = 250_000       # filas por bloque al escribir cada CSV
CHUNKSIZE   = 250_000       # filas por bloque en el baseline secuencial de Pandas
BLOCKSIZE   = '64MB'        # tamaño de partición de Dask (un bloque no cruza archivos)
DIRTY_RATIO = 0.01          # fracción de filas con defectos inyectados a propósito
SEED        = 42
REPEATS     = 3             # repeticiones de cada medición
SCHEDULER   = 'threads'     # 'threads' | 'processes' | 'synchronous'
NUM_WORKERS = os.cpu_count()

REGIONS    = ['norte', 'sur', 'este', 'oeste', 'centro']
CATEGORIES = ['electronica', 'hogar', 'deportes', 'libros', 'ropa', 'alimentos']
GROUPS     = ['region', 'category', 'month']
USECOLS    = ['timestamp', 'region', 'category', 'amount', 'units']
DTYPES     = {'timestamp': 'string', 'region': 'string', 'category': 'string',
              'amount': 'float64', 'units': 'int64'}

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def hardware_info():
    vm = psutil.virtual_memory()
    return {'sistema': platform.platform(), 'procesador': platform.processor() or platform.machine(),
            'nucleos_fisicos': psutil.cpu_count(logical=False), 'nucleos_logicos': psutil.cpu_count(logical=True),
            'ram_total_gb': round(vm.total / 1024**3, 2), 'ram_disponible_gb': round(vm.available / 1024**3, 2)}

print('Python', sys.version.split()[0], '| pandas', pd.__version__, '| dask', dask.__version__, '| numpy', np.__version__)
print(hardware_info())

Python 3.15.1 | pandas 3.1.0 | dask 2026.8.0 | numpy 2.5.4
{'sistema': 'macOS-27.0.1-arm64-arm-64bit-Mach-O', 'procesador': 'arm', 'nucleos_fisicos': 8, 'nucleos_logicos': 8, 'ram_total_gb': 8.0, 'ram_disponible_gb': 1.62}


## Parte 1 · Ingeniero de Datos: lógica Dask y limpieza — César Navarrete

En esta parte generamos los datos, los limpiamos y los agrupamos con Dask.

**De dónde salen los datos.** Generamos `RECORDS` ventas repartidas en `FILES` archivos CSV. Cada archivo se escribe por partes para no tener nunca todo el dataset en memoria. Cada venta tiene `event_id`, `timestamp`, `region`, `category`, `amount`, `units` y `customer_id`.

**Errores a propósito.** Para que la limpieza tuviera algo que hacer, metimos errores en un 1 % de las filas: montos vacíos o negativos, ventas con 0 unidades y regiones que no existen. Además, un 2 % de las filas buenas vienen mal escritas, con espacios o en mayúsculas. Como sabemos cuántas filas dañamos, al final podemos comprobar que la limpieza dejó exactamente las que tenía que dejar.

**Cómo limpiamos** (es igual en Pandas y en Dask):
1. Quitamos espacios y pasamos a minúsculas la región y la categoría.
2. Eliminamos las filas con monto vacío o menor o igual a cero, sin unidades, sin fecha o con una región o categoría que no está en la lista.
3. Sacamos el mes (`YYYY-MM`) de la fecha.

Después agrupamos por región, categoría y mes para obtener el total vendido, el número de ventas y las unidades.

**Notas:**
- **Particiones de Dask:** salieron 16, una por archivo. Cada CSV pesa menos de 64 MB, que es el tamaño de bloque que usa Dask, y un bloque nunca mezcla dos archivos.
- **Columnas y tipos:** solo leemos las 5 columnas que hacen falta para agrupar; `event_id` y `customer_id` no se cargan. Los textos se leen como `string` para que Pandas y Dask usen el mismo tipo. La fecha la dejamos como texto porque para sacar el mes basta con tomar los primeros 7 caracteres.
- **Limitaciones:** los datos son inventados (montos al azar y sin temporadas), así que sirven para medir rendimiento, no para sacar conclusiones de ventas. Con 2 millones de filas (0.13 GB) el dataset todavía cabe en la RAM, por eso Dask gana por poco; si se sube `RECORDS` la diferencia debería notarse más.

In [3]:
def generate_data(folder=DATA_DIR, records=RECORDS, files=FILES, seed=SEED,
                  dirty_ratio=DIRTY_RATIO, gen_chunk=GEN_CHUNK):
    '''Genera los CSV por bloques y guarda data/manifest.json con la "verdad" esperada.'''
    folder.mkdir(parents=True, exist_ok=True)
    for old in folder.glob('events-*.csv'):
        old.unlink()
    base, extra = divmod(records, files)
    start = np.datetime64('2024-01-01T00:00:00')
    span_seconds = 2 * 365 * 24 * 3600
    n_customers = max(1, records // 5)
    invalid_total = 0
    for part in range(files):
        rng = np.random.default_rng([seed, part])
        n_part = base + (1 if part < extra else 0)
        first_id = part * base + min(part, extra)
        path = folder / f'events-{part:04d}.csv'
        written = 0
        while written < n_part:
            n = min(gen_chunk, n_part - written)
            ts = (start + rng.integers(0, span_seconds, n).astype('timedelta64[s]')).astype('datetime64[s]').astype(str)
            ids = np.char.add('evt-', np.char.zfill(np.arange(first_id + written, first_id + written + n).astype(str), 9))
            cust = np.char.add('cust-', np.char.zfill(rng.integers(0, n_customers, n).astype(str), 8))
            df = pd.DataFrame({
                'event_id': ids, 'timestamp': ts,
                'region': rng.choice(REGIONS, n).astype(object),
                'category': rng.choice(CATEGORIES, n).astype(object),
                'amount': np.round(rng.uniform(1, 1200, n), 2),
                'units': rng.integers(1, 9, n),
                'customer_id': cust})
            # Defectos inyectados (filas que la limpieza debe descartar)
            bad = rng.random(n) < dirty_ratio
            kind = rng.integers(0, 4, n)
            df.loc[bad & (kind == 0), 'amount'] = np.nan
            m1 = bad & (kind == 1); df.loc[m1, 'amount'] = -df.loc[m1, 'amount']
            df.loc[bad & (kind == 2), 'units'] = 0
            df.loc[bad & (kind == 3), 'region'] = 'desconocida'
            # Formato sucio pero válido (la limpieza lo normaliza y se conserva)
            fmt = (~bad) & (rng.random(n) < 0.02)
            df.loc[fmt, 'region'] = ' ' + df.loc[fmt, 'region'].str.upper()
            invalid_total += int(bad.sum())
            df.to_csv(path, mode='w' if written == 0 else 'a', header=(written == 0),
                      index=False, float_format='%.2f')
            written += n
        print('Generado:', path)
    manifest = {'records': records, 'files': files, 'seed': seed, 'dirty_ratio': dirty_ratio,
                'invalid_injected': invalid_total, 'expected_valid_rows': records - invalid_total,
                'input_bytes': sum(p.stat().st_size for p in folder.glob('events-*.csv')),
                'generated_at_utc': datetime.now(timezone.utc).isoformat(timespec='seconds')}
    (folder / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    return manifest

def ensure_data(force=False):
    '''Reutiliza los CSV si coinciden con la configuración actual; si no, los regenera.'''
    mf = DATA_DIR / 'manifest.json'
    if mf.exists() and not force:
        m = json.loads(mf.read_text(encoding='utf-8'))
        same = (m['records'], m['files'], m['seed'], m['dirty_ratio']) == (RECORDS, FILES, SEED, DIRTY_RATIO)
        if same and len(list(DATA_DIR.glob('events-*.csv'))) == FILES:
            print('Datos existentes reutilizados (misma configuración).')
            return m
    return generate_data()

manifest = ensure_data()
print(json.dumps(manifest, indent=2))

Datos existentes reutilizados (misma configuración).
{
  "records": 2000000,
  "files": 16,
  "seed": 42,
  "dirty_ratio": 0.01,
  "invalid_injected": 20007,
  "expected_valid_rows": 1979993,
  "input_bytes": 141734541,
  "generated_at_utc": "2026-10-23T20:58:55+00:00"
}


In [4]:
def clean_frame(df):
    '''Limpieza + filtros. Funciona igual sobre DataFrames de Pandas y de Dask.'''
    df = df.assign(region=df['region'].str.strip().str.lower(),
                   category=df['category'].str.strip().str.lower())
    ok = (df['timestamp'].notnull() & df['amount'].notnull() & (df['amount'] > 0) & (df['units'] > 0)
          & df['region'].isin(REGIONS) & df['category'].isin(CATEGORIES))
    df = df[ok]
    return df.assign(month=df['timestamp'].str.slice(0, 7), n=1)

def finalize(agg):
    '''Renombra columnas, calcula avg_amount y ordena. Compartido por Pandas y Dask.'''
    out = agg.rename(columns={'amount': 'total_amount', 'n': 'event_count', 'units': 'total_units'}).reset_index(drop=True)
    out['event_count'] = out['event_count'].astype('int64')
    out['total_units'] = out['total_units'].astype('int64')
    out['total_amount'] = out['total_amount'].astype('float64')
    out['avg_amount'] = out['total_amount'] / out['event_count']
    for g in GROUPS:
        out[g] = out[g].astype(str)
    return out.sort_values(GROUPS).reset_index(drop=True)

def dask_parallel(paths, blocksize=BLOCKSIZE, scheduler=SCHEDULER, num_workers=NUM_WORKERS):
    frame = dd.read_csv(paths, blocksize=blocksize, usecols=USECOLS, dtype=DTYPES)
    npart = frame.npartitions
    rows_read = frame['units'].count()
    agg = clean_frame(frame).groupby(GROUPS)[['amount', 'n', 'units']].sum()
    with dask.config.set(scheduler=scheduler, num_workers=num_workers):
        agg, rows_read = dask.compute(agg, rows_read)
    result = finalize(agg.reset_index())
    return result, {'rows_read': int(rows_read), 'rows_kept': int(result['event_count'].sum()), 'npartitions': int(npart)}

paths = str(DATA_DIR / 'events-*.csv')
dask_result, dask_info = dask_parallel(paths)

# Validación de filas contra la "verdad" del generador
checks_p1 = {
    'filas leídas == RECORDS': dask_info['rows_read'] == manifest['records'],
    'filas válidas == esperadas por el generador': dask_info['rows_kept'] == manifest['expected_valid_rows'],
    'sin valores nulos en agregados': not dask_result.isna().any().any(),
}
print(dask_info)
for k, v in checks_p1.items():
    print('OK ' if v else 'FALLO', k)
assert all(checks_p1.values()), 'La validación de la Parte 1 falló; revisa los datos o la limpieza.'

dask_result.to_csv(RESULTS_DIR / 'agregados_dask.csv', index=False)
print('Guardado:', RESULTS_DIR / 'agregados_dask.csv', f'({len(dask_result)} grupos)')
dask_result.head()

{'rows_read': 2000000, 'rows_kept': 1979993, 'npartitions': 16}
OK  filas leídas == RECORDS
OK  filas válidas == esperadas por el generador
OK  sin valores nulos en agregados
Guardado: results/agregados_dask.csv (720 grupos)


,region,category,month,total_amount,event_count,total_units,avg_amount
0,centro,alimentos,2024-01,1668793.00,2777,12545,600.933741
1,centro,alimentos,2024-02,1564181.30,2569,11678,608.867770
2,centro,alimentos,2024-03,1729586.15,2867,12920,603.273858
3,centro,alimentos,2024-04,1595417.64,2686,12063,593.975294
4,centro,alimentos,2024-05,1755511.99,2906,13028,604.099102


**Revisión cruzada de la Parte 1** · Revisor: DBA NoSQL ([completar nombre])
- [ ] Ejecutó el cuaderno completo en su máquina.
- [ ] Verificó que `rows_kept` coincide con `expected_valid_rows`.
- [ ] Revisó que la limpieza descarta las filas defectuosas y conserva las de formato sucio.
- Observaciones: [completar]

### Aportes del equipo
| Miembro | Rol | Código aportado | Documentación aportada | Módulo que revisó | Commit(s) |
|---|---|---|---|---|---|
| César Navarrete | Ingeniero de Datos | Generación de los datos con errores a propósito, limpieza, agrupación con Dask y revisión de que quedaran las filas correctas | Explicación de la Parte 1: de dónde salen los datos, cómo se limpian, particiones, columnas y limitaciones | Parte 2 | [enlace] |
| [completar nombre] | Arquitecto Cloud | [completar] | [completar] | Parte 3 | [enlace] |
| Alejandro Alvarenga | DBA NoSQL | [completar] | [completar] | Parte 1 | [enlace] |

### Resultados medidos (copiar de la celda anterior)
- **Hardware y sistema:** [completar]
- **Filas y tamaño de entrada:** [completar]
- **Pandas / Dask / Speedup:** [completar] s / [completar] s / [completar] x
- **Equivalencia Pandas–Dask:** [completar]
- **Atlas:** ping [completar] · documentos cargados [completar] · salida de las 3 consultas [completar]

### Evidencia cloud AWS/GCP
- [ ] Recurso desplegado (tipo, región): [completar]
- [ ] Captura de la consola con el recurso en ejecución (fecha visible): [completar]
- [ ] Captura del trabajo ejecutándose en ese recurso: [completar]
- Si no se desplegó infraestructura cloud, se indica en el informe.

### Lista final antes de enviar en Moodle (un solo envío)
- [ ] Los 3 ejecutaron el cuaderno completo.
- [ ] Cada integrante tiene commits propios y revisó el módulo de otro.
- [ ] No hay credenciales en el cuaderno, el repositorio ni el informe.
- [ ] Informe PDF con resultados y capturas.
- [ ] Archivo `.ipynb` o enlace al repositorio.